# hankel-hierarchy: protocol 2 (transient growth), held-out confirmatory run

Settings: **GPU T4 x2 or P100**, **Internet On**. First time: run cells 1–4, stop, and check the smoke line. Real run: **Save Version → Save & Run All**. Held-out first (cell 5). Discovery (cell 6) is exploratory and can run in a later session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__)

In [ ]:
%cd /kaggle/working
!rm -rf hankel-hierarchy && git clone -q https://github.com/CC-Shivansh-Gupta/hankel-hierarchy
%cd hankel-hierarchy
!git log --oneline | head -12
!bash reproduce.sh setup

In [ ]:
# 3. every synthetic suite, protocols 1 and 2
!bash reproduce.sh test

In [ ]:
# 4. SMOKE: random-init network (seed 999), no checkpoint; prints no statistic
!bash reproduce.sh h2-smoke

In [ ]:
# 4b. RESUME from an attached earlier version's output (finished seeds are skipped)
import glob, shutil, os
for s in sorted(glob.glob('/kaggle/input/**/hankel-hierarchy/results2', recursive=True)):
    for f in glob.glob(s + '/**/*', recursive=True):
        if os.path.isfile(f):
            d = os.path.join('results2', os.path.relpath(f, s)); os.makedirs(os.path.dirname(d), exist_ok=True)
            if not os.path.exists(d): shutil.copy2(f, d)
!ls results2/work 2>/dev/null | wc -l

In [ ]:
# 5. HELD-OUT, the confirmatory run
import subprocess
for task in ['reacher-hard', 'finger-turn-hard', 'hopper-stand', 'walker-run', 'quadruped-walk', 'humanoid-run']:
    subprocess.run(['bash', 'reproduce.sh', 'h2-heldout', task], check=True)
    subprocess.run(['bash', '-c', 'tar czf /kaggle/working/results2_partial.tgz results2'], check=True)
subprocess.run(['bash', 'reproduce.sh', 'h2-verdict'], check=True)
subprocess.run(['bash', '-c', 'tar czf /kaggle/working/results2_heldout.tgz results2'], check=True)

In [ ]:
# 6. DISCOVERY (exploratory; optional, can be a separate session)
# !bash reproduce.sh h2-discovery && bash reproduce.sh h2-verdict && tar czf /kaggle/working/results2_all.tgz results2